## 🥁 설정

배포된 모델의 *추론 엔드포인트(Inference endpoint)*와 일치하도록 다음 변수 설정을 변경하세요. 예를 들어:

```
deployed_model_name = "jukebox"
infer_endpoint = "https://jukebox-yyyyyy.apps.cluster-p9k5m.p9k5m.sandboxxxx.opentlc.com"
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import numpy as np
import requests

In [ ]:
deployed_model_name = "jukebox"
infer_endpoint = "<paste-the-link-here>"
infer_url = f"{infer_endpoint}/v2/models/{deployed_model_name}/infer"

    주의: infer_endpoint를 이전 단계에서 배포한 자신의 모델 추론 엔드포인트로 변경했는지 확인하세요.

## 🫡 요청 함수

REST 요청을 구축하고 제출합니다.

In [ ]:
def rest_request(data):
    json_data = {
        "inputs": [ 
           {
                "name": name,
                "shape": [1, 1],
                "datatype": "FP32",
                "data": [data[name]]
            }
            for name in data.keys()
        ]
    }

    response = requests.post(infer_url, json=json_data, verify=True)
    response_dict = response.json()
    return response_dict['outputs'][0]['data']

In [ ]:
# 스케일러와 레이블 인코더 불러오기
with open('models/jukebox/1/artifacts/scaler.pkl', 'rb') as handle:
    scaler = pickle.load(handle)
    
with open('models/jukebox/1/artifacts/label_encoder.pkl', 'rb') as handle:
    label_encoder = pickle.load(handle)

In [ ]:
# 우리가 좋아하는 곡의 특성을 선택합니다
song_properties = pd.read_parquet('../99-data_prep/song_properties.parquet')
favorite_song = song_properties.loc[song_properties["name"]=="Not Like Us"]
favorite_song

In [ ]:
data = favorite_song[['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']]
scaled_data = pd.DataFrame(scaler.transform(data), columns=data.columns)
prediction = rest_request(scaled_data.iloc[0].to_dict())

#### 주어진 곡이 각 국가에서 인기 있을 가능성을 시각화해봅시다!

In [ ]:
plt.figure(figsize = (6, 8))
plt.bar(x = range(len(prediction)),
        height = prediction)
plt.title('Prediction over countries', size = 12, weight = 'bold')
plt.show()

단순히 국가 번호만으로는 큰 의미가 없으므로, 가장 가능성 높은 것을 선택하고 레이블 인코더에 실행하여 국가 코드를 다시 얻습니다

In [ ]:
most_likely_country = np.argmax(prediction)
country_code = label_encoder.inverse_transform([most_likely_country])
print(f"The most likely country is #{most_likely_country} which corresponds to the country code '{country_code[0]}'")

그리고 이것으로 데이터 과학의 내부 루프가 끝입니다!

우리는 다양한 데이터 세트로 작업했고, 탐색적 분석을 했으며, 우리의 곡 발매 전략을 지원할 수 있는 모델을 구축하고 학습했습니다!

이제 매일 새로운 데이터를 받죠? 새로운 데이터로 모델을 학습시키기 위한 자동화가 필요합니다... 그리고 이것이 `파이프라인(pipeline)` 개념이 나오는 부분입니다! 🪄✨

여기의 지침으로 돌아가서 파이프라인을 계속하세요 https://rhoai-mlops.github.io/lab-instructions/#/2-in-the-rhythm-of-data/README